# Semantic Search with Sentence Embeddings

A small hands-on project: searching documents by **meaning** instead of exact keywords.

**What this notebook does**
1. Loads a pretrained embedding model (`all-MiniLM-L6-v2`) using the Sentence Transformers library.
2. Turns 12 short sentences (data protection, cooking, cars, football) into embeddings: 384 numbers per sentence.
3. Looks inside one sentence to see tokens, token vectors and the averaging step.
4. Measures how similar the sentences are to each other (a colour-coded table).
5. Runs a semantic search: a question is embedded and matched against the sentences.

**What it is not:** I did not train or fine-tune any model. The dataset is deliberately tiny so that every number can be inspected.

**Why this matters:** embedding-based search is the foundation of tools that search large collections of documents, and of AI assistants that answer questions from internal files.

*Run in Google Colab (a remote Linux machine), because my local Windows setup blocked a PyTorch file via Smart App Control. October 2026.*

## Core concepts

- **LLM (Large Language Model):** an AI model, built on the transformer architecture, trained on massive amounts of text to understand and generate human language.
- **Transformer:** the neural network architecture behind modern LLMs. Its key idea is "attention": the model weighs how much every word in a sentence should influence every other word, instead of reading strictly left to right.
- **Embedding:** a list of numbers (a vector) that represents the meaning of a piece of text. Texts with similar meaning end up close together in that number space.
- **Pretraining vs. fine-tuning:** pretraining teaches a model general language patterns from huge datasets; fine-tuning trains a pretrained model further on a smaller, specific dataset for a narrower task.
- **This notebook:** uses a small model that someone else already pretrained (nothing is trained here) to build a simple semantic search.

## How a sentence embedding is made

1. **Tokenization:** the sentence is split into tokens (roughly words; rare words are split into smaller pieces).
2. **Vectorization:** every token gets its own vector of 384 numbers. The vector depends on the surrounding words, so "bank" in "river bank" differs from "bank" in "bank account".
3. **Averaging:** the token vectors are averaged, position by position, into ONE vector of 384 numbers: the **sentence embedding**.

**Comparing sentences** is a separate step afterwards: two embeddings are compared with a similarity score. Sentences with similar meaning have similar numbers, so their score is close to 1.

**Why 384?** It is a fixed size chosen by the model's designers. What training taught the model is which numbers to put in those 384 columns.

## Step 1: Install the Sentence Transformers library and load the model

In [ ]:
# Sentence Transformers: a Python library built on top of transformer models.
# Its job is to turn text into embedding vectors.
!pip install -q sentence-transformers

In [1]:
# Load a small pretrained model (BERT-style, called MiniLM). Someone else trained it
# to turn a whole sentence into 384 numbers, so similar meanings get similar numbers.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded


## Step 2: Create my documents

In [2]:
docs = [
    "GDPR sets rules for how companies collect, store and use personal data.",
    "Personal data such as names and IP addresses must be protected under data protection law.",
    "ISO/IEC 27001 is an international standard for managing information security.",
    "Network segmentation limits how far an attacker can move inside a network.",
    "An intrusion detection system monitors network traffic for suspicious activity.",
    "To make pasta, boil salted water and cook the noodles until tender.",
    "A good tomato sauce uses garlic, olive oil and slow-cooked tomatoes.",
    "Bake the bread at high heat until the crust is golden brown.",
    "Electric cars use batteries instead of petrol engines.",
    "Regular oil changes help keep a car engine running smoothly.",
    "The football match ended in a draw after extra time.",
    "The team scored twice in the second half to win the game.",
]
print(len(docs), "documents")

12 documents


## Step 3: Turn the sentences into embeddings

In [3]:
# 12 rows (one per sentence) x 384 columns (the embedding)
embeddings = model.encode(docs)
print(embeddings.shape)

(12, 384)


## Step 4: Look at the new dataset

- **row** = one sentence
- **column** = one of the 384 numbers the model gave that sentence
- one row of 384 numbers = that sentence's **embedding** (also called a vector)

A single number means nothing on its own (column 3 is not "cybersecurity"). The meaning only shows up when a whole row is compared with another whole row. Only the first 10 columns are shown, because 384 would be unreadable.

In [4]:
import pandas as pd

df = pd.DataFrame(embeddings, index=docs)
df.iloc[:, :10]

,0,1,2,3,4,5,6,7,8,9
"GDPR sets rules for how companies collect, store and use personal data.",-0.054898,0.001907,-0.050821,-0.048150,0.022410,0.010062,0.044037,-0.005505,0.015378,-0.073893
Personal data such as names and IP addresses must be protected under data protection law.,-0.050426,0.020951,0.028144,-0.058573,-0.018408,0.062160,0.032599,-0.069861,-0.014276,-0.030960
ISO/IEC 27001 is an international standard for managing information security.,-0.004303,-0.021964,-0.042630,-0.064274,-0.001817,0.013702,0.028045,0.052246,-0.003819,0.002273
Network segmentation limits how far an attacker can move inside a network.,0.002321,0.038662,-0.016121,-0.037726,0.060697,-0.035621,0.032801,0.041666,0.041994,0.007468
An intrusion detection system monitors network traffic for suspicious activity.,-0.014850,0.004942,-0.064007,-0.053869,0.024461,-0.029015,0.129767,-0.011722,0.037745,0.019230
"To make pasta, boil salted water and cook the noodles until tender.",-0.097448,-0.040075,0.005673,0.074368,-0.093533,-0.009778,0.005010,-0.001219,0.002478,-0.110142
"A good tomato sauce uses garlic, olive oil and slow-cooked tomatoes.",-0.092844,0.027377,-0.005110,0.045307,-0.010032,0.025406,-0.003987,-0.054329,0.011122,-0.087649
Bake the bread at high heat until the crust is golden brown.,0.001724,0.019536,0.041468,-0.040841,-0.112120,-0.048361,0.004223,-0.029268,-0.015193,0.062255
Electric cars use batteries instead of petrol engines.,-0.024478,0.120151,0.005425,0.030312,0.011467,0.018483,-0.015748,0.041102,0.032476,-0.005837
Regular oil changes help keep a car engine running smoothly.,-0.098319,0.034030,0.136467,0.030999,0.047526,0.001020,-0.105231,0.015029,-0.005381,-0.149201


## Step 5: Look inside one sentence (tokens, token vectors, averaging)

Below are the three steps from "How a sentence embedding is made" are shown on the first sentence, with real numbers. The last cell checks that averaging the token vectors really gives (almost exactly) the sentence embedding.

In [6]:
sentence = docs[0]
print(sentence)

# 1. Tokenization: the sentence is split into tokens
tokens = model.tokenizer.tokenize(sentence)
print(len(tokens), "tokens:", tokens)

GDPR sets rules for how companies collect, store and use personal data.
15 tokens: ['gdp', '##r', 'sets', 'rules', 'for', 'how', 'companies', 'collect', ',', 'store', 'and', 'use', 'personal', 'data', '.']


In [7]:
# 2. Vectorization: every token gets its own vector of 384 numbers
token_vectors = model.encode(sentence, output_value="token_embeddings")
if isinstance(token_vectors, list):
    token_vectors = token_vectors[0]

print("Token vectors:", tuple(token_vectors.shape), "(tokens x 384)")
# Usually 2 more rows than the token list above: the model adds a special start and end token.

Token vectors: (17, 384) (tokens x 384)


In [8]:
from sentence_transformers import util

# 3. Averaging: the mean of all token vectors, position by position
average_vector = token_vectors.mean(dim=0).cpu()

# Compare with the sentence embedding the model gave us in Step 3
sentence_vector = embeddings[0]
score = float(util.cos_sim(average_vector, sentence_vector))
print("Similarity between my average and the model's sentence embedding:", round(score, 4))
# Close to 1.0 = same direction. (The model also rescales the final vector to a fixed length,
# which does not change which sentences count as similar.)

Similarity between my average and the model's sentence embedding: 1.0


## Step 6: How similar is every sentence to every other sentence?

1.0 = identical meaning, near 0 = unrelated. Darker green = more similar (like conditional formatting in Excel).

In [9]:
similarity = util.cos_sim(embeddings, embeddings).numpy()

labels = ["GDPR rules", "Personal data", "ISO 27001", "Segmentation", "Intrusion detection",
          "Pasta", "Tomato sauce", "Bread", "Electric cars", "Oil change",
          "Football draw", "Football win"]

sim_df = pd.DataFrame(similarity, index=labels, columns=labels).round(2)
sim_df.style.background_gradient(cmap="Greens", vmin=0, vmax=1)

,GDPR rules,Personal data,ISO 27001,Segmentation,Intrusion detection,Pasta,Tomato sauce,Bread,Electric cars,Oil change,Football draw,Football win
GDPR rules,1.000000,0.480000,0.280000,0.060000,0.140000,-0.000000,0.030000,-0.050000,0.040000,0.120000,-0.040000,-0.030000
Personal data,0.480000,1.000000,0.240000,0.220000,0.100000,-0.010000,-0.010000,-0.050000,-0.000000,0.050000,-0.110000,-0.090000
ISO 27001,0.280000,0.240000,1.000000,0.070000,0.200000,-0.020000,-0.020000,-0.050000,0.010000,0.040000,-0.030000,-0.020000
Segmentation,0.060000,0.220000,0.070000,1.000000,0.370000,0.060000,0.130000,-0.050000,-0.070000,-0.030000,-0.010000,-0.010000
Intrusion detection,0.140000,0.100000,0.200000,0.370000,1.000000,0.010000,0.080000,-0.050000,-0.080000,0.040000,0.050000,0.050000
Pasta,-0.000000,-0.010000,-0.020000,0.060000,0.010000,1.000000,0.300000,0.240000,-0.050000,0.070000,-0.040000,0.110000
Tomato sauce,0.030000,-0.010000,-0.020000,0.130000,0.080000,0.300000,1.000000,0.080000,0.040000,0.210000,-0.020000,0.040000
Bread,-0.050000,-0.050000,-0.050000,-0.050000,-0.050000,0.240000,0.080000,1.000000,-0.120000,0.040000,-0.010000,0.060000
Electric cars,0.040000,-0.000000,0.010000,-0.070000,-0.080000,-0.050000,0.040000,-0.120000,1.000000,0.370000,0.000000,-0.020000
Oil change,0.120000,0.050000,0.040000,-0.030000,0.040000,0.070000,0.210000,0.040000,0.370000,1.000000,-0.070000,-0.060000


## Step 7: Semantic search

A question is turned into its own embedding and compared with all 12 sentence embeddings. The highest scores are the best matches. No keyword matching is involved.

In [10]:
def search(question):
    question_embedding = model.encode(question)
    scores = util.cos_sim(question_embedding, embeddings)[0].numpy()
    results = pd.DataFrame({"sentence": docs, "similarity": scores.round(2)})
    return results.sort_values("similarity", ascending=False).reset_index(drop=True)

search("What rules protect personal data?")

,sentence,similarity
0,Personal data such as names and IP addresses m...,0.76
1,"GDPR sets rules for how companies collect, sto...",0.64
2,ISO/IEC 27001 is an international standard for...,0.26
3,Network segmentation limits how far an attacke...,0.22
4,An intrusion detection system monitors network...,0.13
5,Regular oil changes help keep a car engine run...,0.05
6,"A good tomato sauce uses garlic, olive oil and...",0.02
7,"To make pasta, boil salted water and cook the ...",0.00
8,Electric cars use batteries instead of petrol ...,-0.02
9,Bake the bread at high heat until the crust is...,-0.06


In [11]:
# Try a very different question and watch the ranking change
search("How do I cook dinner?")

,sentence,similarity
0,"To make pasta, boil salted water and cook the ...",0.44
1,Bake the bread at high heat until the crust is...,0.34
2,"A good tomato sauce uses garlic, olive oil and...",0.32
3,The team scored twice in the second half to wi...,0.15
4,An intrusion detection system monitors network...,0.02
5,"GDPR sets rules for how companies collect, sto...",0.02
6,Network segmentation limits how far an attacke...,0.02
7,The football match ended in a draw after extra...,0.01
8,Personal data such as names and IP addresses m...,-0.02
9,Regular oil changes help keep a car engine run...,-0.03


## My Observations on Semantic Search

I studied these concepts in my M.Sc. in Data Engineering and later worked with neural-network forecasting models at DemandTex. Since I started applying for AI roles, I have been revisiting them to refresh my hands-on understanding of LLMs and how they are trained.

* I used a pretrained model to power **semantic search**: search that matches by meaning, not exact keywords like Ctrl+F.
* My query **"What rules protect personal data?"** returned the two data-protection sentences as the top matches (0.76 and 0.64), even though neither shares many exact words with the question. The cooking, car and football sentences scored near zero.
* The ISO 27001 and network-security sentences scored 0.26 and 0.22: related, but less than the data-protection sentences.
* Some scores were slightly negative (-0.06). A score below zero means "unrelated", not "opposite".
* Embedding-based search like this is the idea behind tools that search large collections of documents by meaning, which is relevant to document-search products.

## What Surprised Me

I used to picture a language model as a giant ball where similar words live close together: "king" next to "man", "queen" next to "woman". But I couldn't picture how a whole sentence fits into a 384-dimensional space, so I worked through it with a tiny example (3 numbers per word instead of 384, made up for illustration).

Take the sentence "big red car". After tokenization, each word has its own vector:

| Token | Position 1 | Position 2 | Position 3 |
|-------|-----------|-----------|-----------|
| big   | 2 | 4 | 6  |
| red   | 4 | 2 | 8  |
| car   | 6 | 0 | 10 |
| **Average** | **4** | **2** | **8** |

Averaging means: add up the first numbers and divide by 3, then the second numbers, then the third. (2+4+6)/3 = 4, (4+2+0)/3 = 2, (6+8+10)/3 = 8.

So the sentence embedding is [4, 2, 8]: still 3 numbers, the same size as each word's vector, but one list instead of three.

The real model does the same thing with 384 numbers per token (learned during training, not made up). Step 5 above shows it on a real sentence. Sentences are then compared with each other using a similarity score based on their embeddings.

What surprised me is how simple the core idea is once it is broken into small steps.

## Not covered here

- I did not fine-tune the model on domain documents.
- I did not search a large collection (this uses 12 sentences) or stored embeddings in a vector database.
- I did not split long documents into chunks before embedding them.